# Proyecto FUGA IPSS: detección temprana de deserción

**InnovaLab · Instituto Profesional San Sebastián**
Responsable: Fernando Pinto Montecinos · Metodología: **CRISP-DM**

Este notebook documenta el proyecto siguiendo las seis fases de CRISP-DM (*Cross-Industry Standard Process for Data Mining*). Las fases son iterativas: lo que se aprende en una puede obligar a volver a otra anterior.

| Fase | Contenido | Estado |
|---|---|---|
| 1. Comprensión del negocio | Problema, objetivos, criterios de éxito, restricciones | Completada |
| 2. Comprensión de los datos | Inventario, exploración y calidad de las tres fuentes | Completada |
| 3. Preparación de los datos | Limpieza, integración y construcción de la variable objetivo | Completada (etiqueta provisional) |
| 4. Modelado | Predictores, técnicas y diseño de validación | Modelo de referencia (`src/fuga_ipss/modelo.py`); versión definitiva depende de R1 y R2 |
| 5. Evaluación | Métricas, comparación con línea base, sesgo | Preliminar (sección 5.1) |
| 6. Despliegue | Uso por el equipo de seguimiento, monitoreo | Pendiente |
| Anexo | Registro de decisiones, problemas abiertos y solicitudes | Vigente |

**Cómo ejecutarlo:** en orden y de principio a fin. La primera celda de código ubica la raíz del repositorio. Los datos de origen van en `data/raw/` y no se versionan. El notebook genera `data/interim/contactabilidad_log_limpio.pkl` y `data/processed/base_objetivo.pkl`. El mismo proceso está empaquetado para producción en `src/fuga_ipss/` (`python -m fuga_ipss.pipeline`).

**Privacidad:** los estudiantes se identifican con un código pseudonimizado (`EST-…`). No se muestran textos libres ni datos individuales sensibles, y el notebook se versiona sin salidas (filtro `nbstripout`).

**Documentos relacionados:** `docs/03_diccionario_variables/` (diccionario de cada fuente) · `docs/04_documentos_tecnicos/Analisis_Variable_Objetivo.docx` (análisis técnico de la etiqueta) · `docs/04_documentos_tecnicos/Informe_Avance_FUGA_IPSS.docx` (informe para InnovaLab) · `docs/01_informe_ejecutivo/` (resumen ejecutivo).

# 1. Comprensión del negocio

## 1.1 Contexto y problema
El IP San Sebastián registra una deserción relevante de estudiantes. Hoy la gestión es principalmente **reactiva**: el equipo de seguimiento contacta a los estudiantes (registro en el SGE) cuando ya muestran señales de abandono, como inasistencia, evaluaciones sin entregar o el aviso de que van a suspender.

## 1.2 Objetivo del negocio
Identificar **tempranamente** a los estudiantes con mayor riesgo de desertar, para que el equipo de seguimiento priorice sus acciones de retención **antes** de que la decisión de irse esté tomada.

## 1.3 Objetivo analítico
Construir un modelo de **clasificación binaria** que estime la probabilidad de deserción (`DESERCION`) de cada estudiante, usando solo información disponible al momento de la predicción:
- **Momento A, al ingreso:** encuesta de caracterización.
- **Momento B, al cierre del primer periodo:** encuesta, primeras notas y primeros contactos.

## 1.4 Criterios de éxito
| Tipo | Criterio | Meta |
|---|---|---|
| Negocio | El equipo de seguimiento recibe una lista priorizada y útil para su trabajo | Validar con InnovaLab |
| Analítico | Supera claramente a la línea base (tasa de deserción ≈ 30%) en PR-AUC | A definir con InnovaLab |
| Analítico | Concentra a los desertores entre los estudiantes de mayor riesgo (recall en el 20% de mayor riesgo) | A definir con InnovaLab |
| Analítico | Alerta con anticipación al evento (requiere fechas: solicitud R1) | Meses de anticipación |
| Ético | Sin diferencias injustificadas de desempeño por variables sensibles | Revisión en la fase 5 |

## 1.5 Actores
| Actor | Rol |
|---|---|
| InnovaLab, coordinación del área | Mandante; valida objetivos y prioridades |
| Registro Académico | Fuente de datos oficiales (estados, fechas, notas) |
| Equipo de seguimiento y retención | Usuario final de las alertas |
| Fernando Pinto Montecinos | Asesor de ML: análisis y modelamiento |

## 1.6 Restricciones, supuestos y riesgos
- **Privacidad:** los datos incluyen información sensible (salud, discapacidad, pueblo originario, ingresos). Se trabaja con identificadores pseudonimizados y no se publican datos individuales.
- **Fuga de información (leakage):** solo se pueden usar predictores anteriores al momento de predicción.
- **Supuesto vigente:** la fecha de corte del estado académico es 2026-08-31 (por confirmar; solicitud R2).
- **Riesgo principal:** el estado académico no trae fecha del evento. Sabemos **quién** desertó, pero no **cuándo** (ver Anexo).

# 2. Comprensión de los datos

## 2.1 Inventario de fuentes
| Fuente | Archivo(s) | Granularidad | Cobertura | Volumen | Llave | Diccionario |
|---|---|---|---|---|---|---|
| Contactabilidad SGE | `Contactabilidad_SGE.xlsx` | Interacción funcionario-estudiante | ene-2024 a ago-2026 | 43.215 interacciones | `RUT_ALUMNO` | `Diccionario_Contactabilidad.xlsx` |
| Encuesta de caracterización | `encuesta_caracterizacion_2024/2025/2026.xlsx` | Respuesta de un estudiante al ingresar | Cohortes 2024, 2025 y 2026 | 3.317 respuestas, 365 variables | `token` | `Diccionario_Encuesta_Caracterizacion.xlsx` |
| Rendimiento académico | `rendimiento_academico_2024-2025.xlsx` | Nota de una evaluación | 2024-P1 a 2025-P1 | 70.000 notas, 2.062 estudiantes | `RUT` | `Diccionario_Rendimiento_Academico.xlsx` |

Las tres llaves contienen el **mismo identificador pseudonimizado** (`EST-…`), así que las fuentes se pueden cruzar a nivel de estudiante.

### Carga de librerías y datos

In [ ]:
# Ubicar la raíz del repositorio para que las rutas relativas funcionen desde notebooks/
import os
from pathlib import Path
raiz = Path.cwd()
while not (raiz / 'pyproject.toml').exists() and raiz != raiz.parent:
    raiz = raiz.parent
os.chdir(raiz)
print('Raíz del proyecto:', raiz)


In [ ]:
#####################################
"""
IMPORTACION DE LIBRERIAS

"""
########################################

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re, unicodedata



In [ ]:
contactabilidad = pd.read_excel('data/raw/Contactabilidad_SGE.xlsx')
caracterizacion_2024 = pd.read_excel('data/raw/encuesta_caracterizacion_2024.xlsx')
caracterizacion_2026 = pd.read_excel('data/raw/encuesta_caracterizacion_2026.xlsx')
caracterizacion_2025 = pd.read_excel('data/raw/encuesta_caracterizacion_2025.xlsx')

## 2.2 Contactabilidad: exploración inicial
Primera revisión de estructura, nulos, duplicados y valores frecuentes. El correlativo `Número` se descarta de inmediato porque no aporta información.

In [ ]:
contactabilidad.head(1)

In [ ]:
contactabilidad.drop(columns=['Número'], inplace=True)

In [ ]:
contact = contactabilidad.copy()
print(contact.shape)
contact.info()
print(contact.isna().sum())
print("Duplicados exactos:", contact.duplicated().sum())

for c in ['Asunto', 'CODCARR', 'TIPO_COMUNICACION', 'Rut_responsable']:
    print(f"\n--- {c}: {contact[c].nunique()} únicos")
    print(contact[c].value_counts().head(15))

print("\nRango fechas:", contact['FECHA_COMUNICO'].min(), "→", contact['FECHA_COMUNICO'].max())
print("Alumnos únicos:", contact['RUT_ALUMNO'].nunique())
print(contact.groupby('RUT_ALUMNO').size().describe())

**Lectura:** es un **registro de eventos**, no una tabla por estudiante, así que habrá que agregarlo. El asunto aporta poco (casi todo dice "Académico") y la información útil está en el texto libre de las observaciones. La limpieza y los hallazgos completos están en la sección 3.1.

## 2.3 Encuesta de caracterización: exploración inicial
Los tres archivos son el mismo instrumento (exportado de LimeSurvey) aplicado a cada cohorte de ingreso. El detalle de las 365 variables, sus secciones y los cambios de redacción entre años está en el diccionario.

In [ ]:
caracterizacion_24 = caracterizacion_2024.copy()
caracterizacion_24.head()

In [ ]:
caracterizacion_24.info()

In [ ]:
caracterizacion_2026['groupTime509. Temporización del grupo: DISPOSICIÓN HACIA LA EDUCACIÓN SUPERIOR'].quantile([0.5,0.75])

**Lectura:** cada archivo tiene unas 330–365 columnas: preguntas, sub-ítems de selección múltiple, escalas Likert y tiempos de respuesta (paradatos). Muchas preguntas son condicionales, así que un nulo suele significar "no aplicaba". Los tres archivos traen además `Carrera`, `Jornada / modalidad` y `Estado académico`, que es la base de la variable objetivo (sección 3.4).

## 2.4 Rendimiento académico: cobertura
Una fila es la nota de una evaluación. Se revisa qué periodos cubre el extracto. Se descartan las columnas anonimizadas o constantes (`NOMBRE`, `PATERNO`, `MATERNO`, `ESTADO_ASIGNATURA`).

In [ ]:
rend = pd.read_excel('data/raw/rendimiento_academico_2024-2025.xlsx')
rend = rend.drop(columns=['Número', 'NOMBRE', 'PATERNO', 'MATERNO', 'ESTADO_ASIGNATURA'])  # anonimizadas o constantes
rend['RUT'] = rend['RUT'].astype(str).str.strip().str.upper()
rend['FECHAINICIO'] = pd.to_datetime(rend['FECHAINICIO'], dayfirst=True)
rend['FECHAFIN'] = pd.to_datetime(rend['FECHAFIN'], dayfirst=True)

rend.groupby(['ANO', 'PERIODO']).agg(estudiantes=('RUT', 'nunique'),
                                     inicio=('FECHAINICIO', 'min'),
                                     fin=('FECHAFIN', 'max'))

**Lectura:** el extracto va de 2024-P1 a 2025-P1. Los periodos no son equivalentes entre regímenes (trimestral, bimestral, semestral), y de 2025 solo está el primer periodo.

## 2.5 Resumen de calidad de los datos
| Fuente | Problema | Tratamiento |
|---|---|---|
| Contactabilidad | `TIPO_COMUNICACION` dice "LLAMADO" en el 98% de los casos, pero las notas muestran WhatsApp y correo | Canal inferido desde el texto (`TX_CANAL_*`) |
| Contactabilidad | Un cuarto de los registros son plantillas o correos masivos | Marcados con `ES_PLANTILLA`; no generan señales |
| Contactabilidad | 115 registros sin carrera; 7% de los estudiantes asociado a más de una carrera | Imputación por moda del estudiante |
| Encuesta | La redacción de algunas preguntas cambió entre años | Revisar en el diccionario antes de juntar años |
| Encuesta | Ingreso familiar con valores extremos (hasta 4.000 millones) | Tratar antes de modelar |
| Rendimiento | Notas bimodales: muchas 1,0 (no rinde o no entrega) y muchas 7,0 | Se resume con `pct_nota_minima` |
| Rendimiento | La cohorte 2026 casi no tiene notas | Solicitud R5 |
| Todas | No existe una fecha de deserción | Solicitud R1 (ver Anexo) |

# 3. Preparación de los datos

## 3.1 Limpieza de contactabilidad
Normalización de texto, imputación de carrera, tratamiento de duplicados, detección de plantillas y construcción de indicadores `TX_*` a partir del texto de las observaciones. El resultado se guarda en `contactabilidad_log_limpio.pkl`.

In [ ]:
def norm_txt(s):
    if pd.isna(s):
        return np.nan
    s = str(s).replace('\\n', ' ').replace('\n', ' ')
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    return re.sub(r'\s+', ' ', s).strip().upper()

contact['ASUNTO'] = (contact['Asunto'].map(norm_txt)
                       .str.replace(r'[^A-Z0-9 /-]', '', regex=True).str.strip())
contact['OBS_LIMPIA'] = contact['OBSERVACION'].map(norm_txt)

for c in ['RUT_ALUMNO', 'CODCARR', 'Rut_responsable', 'TIPO_COMUNICACION']:
    contact[c] = contact[c].astype('string').str.strip().str.upper()

contact['FECHA_COMUNICO'] = pd.to_datetime(contact['FECHA_COMUNICO'], errors='coerce')

f = contact['FECHA_COMUNICO']
inc = (f.dt.day != contact['DIA']) | (f.dt.month != contact['MES']) | (f.dt.year != contact['ANIO'])
print("Fechas inconsistentes:", inc.sum(), "| Fechas nulas:", f.isna().sum())

# Imputar CODCARR con la moda del mismo alumno
moda = (contact.dropna(subset=['CODCARR'])
               .groupby('RUT_ALUMNO')['CODCARR']
               .agg(lambda s: s.mode().iat[0]))
contact['CODCARR'] = contact['CODCARR'].fillna(contact['RUT_ALUMNO'].map(moda))
print("CODCARR nulos tras imputar:", contact['CODCARR'].isna().sum())

# ¿Alumnos con más de una carrera? Importa para el cruce con caracterización
print("Alumnos con >1 carrera:", (contact.groupby('RUT_ALUMNO')['CODCARR'].nunique() > 1).sum())

contact = contact.drop(columns=['Asunto', 'OBSERVACION', 'Nombre_responsable',
                                'DIA', 'MES', 'ANIO'])

antes = len(contact)
contact = contact.drop_duplicates(subset=['RUT_ALUMNO', 'FECHA_COMUNICO', 'ASUNTO',
                                          'TIPO_COMUNICACION', 'Rut_responsable'])
print("Duplicados eliminados:", antes - len(contact))

In [ ]:
def categoria(a):
    if pd.isna(a):                                    return 'OTRO'
    if 'NO CONTESTA' in a or 'ERRONEO' in a:          return 'NO_CONTACTO'
    if 'RETENCION' in a:                              return 'RETENCION'
    if 'FINANZ' in a or 'FINANCIAMIENTO' in a:        return 'FINANZAS'
    if 'NO PARTICIPACION' in a or 'EVALUACION' in a:  return 'EVALUACION'
    if 'EXPERIENCIA' in a:                            return 'EXPERIENCIA'
    if 'ACADEM' in a or 'INSCRIPCION' in a:           return 'ACADEMICO'
    if 'IMPORTANTE' in a:                             return 'IMPORTANTE'
    return 'OTRO'

contact['CAT_ASUNTO'] = contact['ASUNTO'].map(categoria)
contact['ES_SEGUIMIENTO'] = contact['ASUNTO'].str.contains('SEGUIMIENTO', na=False).astype(int)

print(contact['CAT_ASUNTO'].value_counts())
# Revisar todo lo que no cayó en ninguna categoría
print(contact.loc[contact['CAT_ASUNTO'] == 'OTRO', 'ASUNTO'].value_counts())

In [ ]:
tmp = contactabilidad.copy()
tmp['ASUNTO'] = (tmp['Asunto'].map(norm_txt)
                   .str.replace(r'[^A-Z0-9 /-]', '', regex=True).str.strip())
claves = ['RUT_ALUMNO', 'FECHA_COMUNICO', 'ASUNTO', 'TIPO_COMUNICACION', 'Rut_responsable']
dup = tmp.duplicated(subset=claves, keep=False)

print(tmp.loc[dup, 'ASUNTO'].map(categoria).value_counts())
print(tmp[dup].groupby(claves).size().value_counts().sort_index())   # cuántas repeticiones por grupo
print(tmp.loc[dup, 'OBSERVACION'].head(10).tolist())                 # ¿el texto cambia o es idéntico?

In [ ]:
contact = contact.drop_duplicates(subset=['RUT_ALUMNO', 'FECHA_COMUNICO', 'ASUNTO',
                                          'TIPO_COMUNICACION', 'Rut_responsable', 'OBS_LIMPIA'])

In [ ]:
contact['OBS_LIMPIA'] = contact['OBS_LIMPIA'].str.replace(r'\S+@\S+', 'EMAIL', regex=True)

# Plantilla = texto idéntico repetido muchas veces
freq = contact['OBS_LIMPIA'].map(contact['OBS_LIMPIA'].value_counts())
contact['ES_PLANTILLA'] = (freq >= 20).astype(int)
print("Proporción plantillas:", contact['ES_PLANTILLA'].mean().round(3))
print(contact.loc[contact['ES_PLANTILLA'] == 1, 'OBS_LIMPIA'].str[:120].value_counts().head(10))

patrones = {
    'NO_CONTESTA':    r'NO CONTEST|SIN EXITO|NO RESPONDE|NO LE LLEGA|ERRONEO|BUZON|APAGADO',
    'RESPONDE':       r'(ALUMN[OA]|ESTUDIANTE) (CONTESTA|RESPONDE|INDICA|SENALA|COMENTA)',
    'INASISTENCIA':   r'NO (HA|A) ASISTIDO|INASISTENCIA|NO ASISTE|AUSENT',
    'SIN_ENTREGAS':   r'ENTREGA|\bEVA ?\d|NO PARTICIPA',
    'INSCRIPCION':    r'INSCRI|TOMA DE RAMOS',
    'MATRICULA':      r'MATRICUL',
    'FINANCIERO':     r'FINANCI|DEUDA|PAGO|ARANCEL|BECA|\bCAE\b|GRATUIDAD',
    'DESISTE':        r'DESIST|SUSPEN|ELIMINACION|RETIR|CONGEL|ABANDON',
    'CONTINUA':       r'QUIERE CONTINUAR|SEGUIR ESTUDIANDO|PONERSE AL DIA|CONTINUAR',
    'CANAL_WSP':      r'WTSP|WSP|WHATSAPP|WASAP',
    'CANAL_CORREO':   r'CORREO',
}
for k, p in patrones.items():
    contact[f'TX_{k}'] = contact['OBS_LIMPIA'].str.contains(p, regex=True, na=False).astype(int)

# En las plantillas estas palabras están siempre presentes: no son señal
for k in ['DESISTE', 'FINANCIERO', 'CONTINUA']:
    contact.loc[contact['ES_PLANTILLA'] == 1, f'TX_{k}'] = 0

tx = [c for c in contact if c.startswith('TX_')]
print(contact[tx].mean().round(3).sort_values(ascending=False))

# Validación manual: 3 ejemplos aleatorios por indicador
for c in tx:
    print(f"\n=== {c}")
    for t in contact.loc[contact[c] == 1, 'OBS_LIMPIA'].sample(3, random_state=1):
        print("  -", t[:150])

In [ ]:
patrones = {
    'NO_CONTESTA':   r'\bNO CONTEST|\bNO RESPONDE|SIN EXITO|\bNO LE LLEGA|ERRONEO|NO CORRESPONDE AL|BUZON|APAGADO|SE INTENT[OA] (?:CONTACT|COMUNIC)',
    'RESPONDE':      r'(?:ALUMN[OA]|ESTUDIANTE) (?:CONTESTA|RESPONDE|INDICA|SENALA|COMENTA|CONFIRMA)',
    'INICIA_ALUMNO': r'(?:ALUMN[OA]|ESTUDIANTE) SE CONTACTA',
    'INASISTENCIA':  r'\bNO (?:HA |A )?ASISTID|INASISTENCIA|\bNO ASISTE|AUSENT|NO HA (?:ENTRADO|INGRESADO|CONECTADO)',
    'SIN_ENTREGAS':  r'\bENTREGAS?\b|INCUMPLIMIENTO|\bEVA ?\d|NO PARTICIPA',
    'INSCRIPCION':   r'INSCRI|TOMA DE RAMOS',
    'NO_MATRICULADO':r'\bNO SE (?:A |HA )?MATRICUL|\bNO (?:ESTA|SE ENCUENTRA) MATRICUL|AUN NO (?:SE )?MATRICUL',
    'MATRICULADO':   r'\bSE MATRICULO|YA (?:SE )?MATRICUL|YA ESTA MATRICUL',
    'FINANCIERO':    r'FINANCI|DEUDA|MOROS|\bPAGOS?\b|ARANCEL|BECA|\bCAE\b|GRATUIDAD|CREDITO',
    'DESISTE':       r'DESIST|SUSPEN|ELIMINACION|\bRETIR|CONGEL|ABANDON',
    'CONTINUA':      r'(?:QUIERE|DESEA|VA A|INDICA QUE) (?:SEGUIR|CONTINUAR)|PONERSE AL DIA|PONDRA AL DIA',
    'NO_CONTINUA':   r'\bNO (?:VA A |QUIERE |DESEA |PODRA )?(?:CONTINUAR|SEGUIR)\b|\bNO CONTINUARA|\bNO SEGUIRA',
    'CANAL_WSP':     r'WTSP|\bWSP|WHATSAPP|WASAP|\bWTS\b',
    'CANAL_CORREO':  r'CORREO(?! INSTITUCIONAL)',
}

contact = contact.drop(columns=[c for c in contact if c.startswith('TX_')])
for k, p in patrones.items():
    contact[f'TX_{k}'] = contact['OBS_LIMPIA'].str.contains(p, regex=True, na=False).astype(int)

for k in ['DESISTE', 'FINANCIERO', 'CONTINUA', 'NO_CONTINUA']:
    contact.loc[contact['ES_PLANTILLA'] == 1, f'TX_{k}'] = 0

tx = [c for c in contact if c.startswith('TX_')]
print(contact[tx].mean().round(3).sort_values(ascending=False))

# Validación de los indicadores que cambiaron, ahora con texto completo
for c in ['TX_NO_CONTESTA', 'TX_CONTINUA', 'TX_NO_CONTINUA', 'TX_SIN_ENTREGAS',
          'TX_NO_MATRICULADO', 'TX_INICIA_ALUMNO', 'TX_DESISTE']:
    print(f"\n=== {c}  (n={contact[c].sum()})")
    for t in contact.loc[contact[c] == 1, 'OBS_LIMPIA'].sample(4, random_state=7):
        print("  -", t[:250])

In [ ]:
# 1. Ampliar la detección de plantillas a correos de tutores
voz_staff = contact['OBS_LIMPIA'].str.contains(r'^ESTIMAD|JUNTO CON SALUDAR', regex=True, na=False)
contact['ES_PLANTILLA'] = ((contact['ES_PLANTILLA'] == 1) | voz_staff).astype(int)

# 2. Texto en voz del alumno: todo lo anterior a la respuesta del funcionario
contact['OBS_ALUMNO'] = contact['OBS_LIMPIA'].str.split('SE LE RESPONDE', n=1, regex=False).str[0]

# 3. Ajustes de patrones
patrones['NO_CONTINUA'] = r'\bNO (?:VA A |QUIERE |DESEA |PODRA |PUEDE )?(?:CONTINUAR|SEGUIR)\b|\bNO CONTINUARA|\bNO SEGUIRA'
patrones['TITULACION']  = r'TITULACION|\bTITULAR|EGRESAD'

voz_alumno = ['DESISTE', 'CONTINUA', 'NO_CONTINUA']
antes = contact[[f'TX_{k}' for k in voz_alumno]].sum()

for k, p in patrones.items():
    col = 'OBS_ALUMNO' if k in voz_alumno else 'OBS_LIMPIA'
    contact[f'TX_{k}'] = contact[col].str.contains(p, regex=True, na=False).astype(int)

for k in voz_alumno + ['FINANCIERO']:
    contact.loc[contact['ES_PLANTILLA'] == 1, f'TX_{k}'] = 0

print("Proporción plantillas:", contact['ES_PLANTILLA'].mean().round(3))
print(pd.DataFrame({'antes': antes,
                    'despues': contact[[f'TX_{k}' for k in voz_alumno]].sum()}))
print("Alumnos con mención de titulación:",
      contact.loc[contact['TX_TITULACION'] == 1, 'RUT_ALUMNO'].nunique())

# 4. Guardar el log limpio
contact = contact.drop(columns=['OBS_ALUMNO'], errors='ignore')
contact.to_pickle('data/interim/contactabilidad_log_limpio.pkl')
try:
    contact.to_parquet('data/interim/contactabilidad_log_limpio.parquet', index=False)
except Exception as e:
    print("Parquet no disponible, quedó solo el pickle:", type(e).__name__)
print(contact.shape)

In [ ]:
contact.to_pickle('data/interim/contactabilidad_log_limpio.pkl')
print(contact.shape)

### Hallazgos de la limpieza de contactabilidad

Estructura y volumen

El dataset es un registro de interacciones entre funcionarios y alumnos, no una tabla por alumno. Para el modelo hay que agregarlo después.
Tras la limpieza quedan 38.616 interacciones de 6.552 alumnos, entre enero de 2024 y agosto de 2026.
Cada alumno tiene en promedio unos 6 contactos. La mitad tiene 5 o menos, y hay casos extremos de hasta 67.
462 alumnos (7%) aparecen asociados a más de una carrera. Por eso el cruce con los otros datasets debe hacerse por alumno y carrera, no solo por alumno.

Calidad de los datos

Las fechas están completas y son consistentes. Las columnas de día, mes y año eran redundantes y se eliminaron.
Solo 115 registros no tenían carrera, y se completaron con la carrera que el mismo alumno tiene en sus otros registros.
Lo que parecían duplicados (más de 4.500 filas) eran en realidad interacciones distintas del mismo día: un llamado sin respuesta, luego un WhatsApp, luego la respuesta del alumno. Se conservaron, porque esa secuencia es información valiosa.
El tipo de comunicación registrado no es confiable: dice “llamado” en el 98% de los casos, pero las notas muestran mucho uso de WhatsApp y correo.

Dónde está la información útil

El asunto del contacto aporta poco: el 86% dice simplemente “Académico”.
La información relevante está en las observaciones escritas por los ejecutivos. Ahí se registra el motivo del contacto, si el alumno respondió y lo que dijo.
Un cuarto de los registros son plantillas o correos masivos (bienvenidas, encuestas, recordatorios de inscripción) y hubo que separarlos de las notas personalizadas. Si no, palabras como “desistir” o “problemas financieros”, que aparecen en todas las plantillas, se habrían leído como señales del alumno.

Señales detectadas en el texto

El 27% de los contactos registra que el alumno no contestó. Probablemente es la señal más potente de desconexión.
El 16% son contactos iniciados por el propio alumno, generalmente por WhatsApp. Es una señal de compromiso, aunque a veces lo que el alumno plantea es un problema.
El 12% se refiere a evaluaciones no entregadas y cerca del 3% a inasistencia.
El 8% menciona temas financieros: deudas, morosidad, becas, CAE o pagos.
Hay manifestaciones explícitas de intención: unas 800 menciones de suspender o abandonar, unas 300 de no querer continuar y unas 450 de querer continuar o ponerse al día.

Alertas para el modelo

La institución trabaja por trimestres, no por semestres. La unidad del modelo debería ser alumno-carrera-trimestre.
Riesgo de fuga de información (leakage). Varios contactos ocurren porque el alumno ya está desertando: por ejemplo, las llamadas a alumnos que no se han matriculado, los contactos de retención o los alumnos que avisan que van a suspender. Esas variables se verían excelentes al entrenar, pero no sirven para detectar a tiempo. Solo se pueden usar contactos anteriores a la fecha de corte de cada trimestre.
533 alumnos (8%) mencionan titulación, es decir, terminaron la carrera. Hay que tratarlos como egresados al definir el target, para que no queden mal etiquetados como desertores.
Los textos contienen nombres de alumnos y correos de funcionarios. Se conservan solo para trazabilidad y no entran al modelo.

Pendiente

La agregación final por alumno y trimestre depende del calendario académico y de la definición de deserción, que vendrán del dataset de rendimiento académico.

## 3.2 Unificación de la encuesta por cohorte

Los tres archivos son el **mismo instrumento** aplicado a quienes ingresan cada año, así que cada archivo se trata como una **cohorte de ingreso**. Por ahora solo se toman las columnas necesarias para definir la variable objetivo; las preguntas de la encuesta se incorporarán como predictores en la fase 4.
La llave es `token`, el mismo identificador pseudonimizado (`EST-…`) que `RUT` en rendimiento y `RUT_ALUMNO` en contactabilidad.

In [ ]:
import matplotlib.pyplot as plt
from itertools import combinations

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.edgecolor': '#888888'})

COLS_CLAVE = {'token. Código de acceso': 'token', 'submitdate. Fecha de envío': 'fecha_encuesta',
              'Carrera': 'carrera', 'Jornada / modalidad': 'jornada', 'Estado académico': 'estado'}
encuestas = {2024: caracterizacion_2024, 2025: caracterizacion_2025, 2026: caracterizacion_2026}

partes = []
for anio, df in encuestas.items():
    p = df[list(COLS_CLAVE)].rename(columns=COLS_CLAVE)
    p['cohorte'] = anio
    partes.append(p)
enc = pd.concat(partes, ignore_index=True)
enc['token'] = enc['token'].astype(str).str.strip().str.upper()
enc['fecha_encuesta'] = pd.to_datetime(enc['fecha_encuesta'], errors='coerce')

print(enc.shape, '| tokens únicos:', enc['token'].nunique())
enc.groupby('cohorte').agg(respuestas=('token', 'size'),
                           primera_respuesta=('fecha_encuesta', 'min'),
                           ultima_respuesta=('fecha_encuesta', 'max'))

In [ ]:
# ¿Hay estudiantes que respondieron en más de un año?
for a, b in combinations(encuestas, 2):
    comunes = set(enc.loc[enc.cohorte == a, 'token']) & set(enc.loc[enc.cohorte == b, 'token'])
    print(f'{a} ∩ {b}: {len(comunes)} estudiantes')

**Lectura:** casi no hay estudiantes repetidos entre años (14 en total), lo que confirma que cada archivo es una cohorte de ingreso distinta.
Para los repetidos se conserva la **primera** cohorte (año de ingreso).

## 3.3 Agregación del rendimiento por estudiante
Se resume cada estudiante con indicadores de desempeño y de desconexión académica.

In [ ]:
rend_alumno = rend.groupby('RUT').agg(
    primera_clase=('FECHAINICIO', 'min'),
    ultima_clase=('FECHAFIN', 'max'),
    n_evaluaciones=('Nota', 'size'),
    nota_promedio=('Nota', 'mean'),
    pct_reprobadas=('ESTADO', lambda s: (s == 'R').mean()),
    pct_nota_minima=('Nota', lambda s: (s <= 1.0).mean()),
    cursa_2025=('ANO', lambda s: (s == 2025).any()),
)
print('Estudiantes en rendimiento:', len(rend_alumno))
print('Cruce con cada cohorte de la encuesta:',
      {a: len(set(rend_alumno.index) & set(enc.loc[enc.cohorte == a, 'token'])) for a in encuestas})
rend_alumno.describe().T.round(2)

**Lectura:**
- El extracto cubre de 2024-P1 a 2025-P1. La mayoría de los estudiantes parte el 25-03-2024, así que corresponde sobre todo a la **cohorte 2024**: 561 de los 600 encuestados de 2024 están en rendimiento.
- Las notas son bimodales: muchas 1,0 (no rinde o no entrega) y muchas 7,0. `pct_nota_minima` resume bien la desconexión académica.
- La cohorte 2026 casi no tiene notas (17 estudiantes), así que para ella el rendimiento todavía no sirve como predictor.

## 3.4 Construcción y validación de la variable objetivo

> **Nota CRISP-DM:** la variable objetivo se construye aquí y no en la fase 2, porque necesita las tres fuentes limpias e integradas. Lo que se aprendió en esta sección se devolvió a la fase 1 (riesgos y supuestos) y al Anexo.

Ninguna fuente trae una columna llamada "deserción". Hay cuatro formas de construirla:

| Opción | Definición | Ventaja | Problema |
|---|---|---|---|
| **A. Ausencia en rendimiento** | Tiene notas en un periodo y no en el siguiente | Conductual, objetiva | El extracto termina en 2025-P1; no distingue deserción de pausa, régimen distinto o corte del extracto |
| **B. Señales de contactabilidad** | `TX_DESISTE`, `TX_NO_CONTINUA`, `TX_NO_MATRICULADO` | Cubre hasta ago-2026 | Solo existe para quienes fueron contactados; sesgo de registro; circular si también es predictor |
| **C. Registro oficial** | Lista de retiros, eliminaciones o no matriculados del SGE con fecha | Es la verdad institucional | Hay que solicitarlo |
| **D. `Estado académico` de la encuesta** | Estado del estudiante anexado a los tres archivos | Ya está disponible, viene del sistema académico, cubre las 3 cohortes | No trae fecha del estado ni fecha de corte |

A continuación se evalúa **D** como etiqueta principal y se valida contra A y B.

### 3.4.1 Estado académico por cohorte

In [ ]:
print(pd.crosstab(enc['cohorte'], enc['estado'], margins=True, margins_name='Total'))
(pd.crosstab(enc['cohorte'], enc['estado'], normalize='index') * 100).round(1)

In [ ]:
GRUPO_ESTADO = {'VIGENTE': 'Vigente', 'TITULADO': 'Titulado / egresado', 'EGRESADO': 'Titulado / egresado',
                'SUSPENDIDO': 'Suspendido', 'ELIMINADO': 'Eliminado', 'SIN DATO': 'Sin dato'}
ORDEN = ['Vigente', 'Titulado / egresado', 'Suspendido', 'Eliminado', 'Sin dato']
COLORES = {'Vigente': '#C9CED6', 'Titulado / egresado': '#4C78A8', 'Suspendido': '#F2A541',
           'Eliminado': '#B8433E', 'Sin dato': '#EDEDED'}

tab = (pd.crosstab(enc['cohorte'], enc['estado'].map(GRUPO_ESTADO), normalize='index')
         .reindex(columns=ORDEN, fill_value=0) * 100)

fig, ax = plt.subplots(figsize=(8, 2.6))
izq = np.zeros(len(tab))
for est in ORDEN:
    v = tab[est].values
    ax.barh(tab.index.astype(str), v, left=izq, color=COLORES[est], label=est, edgecolor='white', linewidth=2)
    for y, (x0, w) in enumerate(zip(izq, v)):
        if w >= 6:
            ax.text(x0 + w / 2, y, f'{w:.0f}%', ha='center', va='center', fontsize=9,
                    color='white' if est in ('Eliminado', 'Titulado / egresado') else '#333333')
    izq += v
ax.set_xlim(0, 100); ax.set_xlabel('% de la cohorte'); ax.invert_yaxis()
ax.set_title('Estado académico al corte del extracto, por cohorte de ingreso', loc='left', fontsize=11)
ax.legend(ncol=5, loc='upper left', bbox_to_anchor=(0, -0.3), frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

**Lectura:**
- `Estado académico` existe en los tres archivos y su distribución cambia mucho por cohorte. La cohorte 2024 ya tiene un 24% de **titulados**, así que el estado **no** es el del año de ingreso: es una foto tomada en una fecha de corte común y posterior (probablemente 2026).
- Entonces cada cohorte tuvo un **tiempo de exposición distinto**: 2024 lleva más de 2 años, 2026 solo meses. Las tasas no son comparables entre cohortes sin controlar el tiempo.
- La cohorte 2026 tiene **más** eliminados (30%) que la 2025 (17%), aunque lleva menos tiempo. Eso sugiere abandono muy temprano o no inicio de clases (matriculados que nunca empiezan), o bien que la fecha de corte no es la misma en los tres archivos. **Hay que confirmarlo con Registro Académico.**

### 3.4.2 Validación con rendimiento académico

In [ ]:
estado_alumno = enc.sort_values('cohorte').drop_duplicates('token').set_index('token')[['estado', 'cohorte']]
val_rend = rend_alumno.join(estado_alumno, how='inner')
print('Estudiantes con notas y estado académico:', len(val_rend))

(val_rend.groupby('estado')
         .agg(n=('nota_promedio', 'size'), nota_promedio=('nota_promedio', 'mean'),
              pct_reprobadas=('pct_reprobadas', 'mean'), pct_nota_minima=('pct_nota_minima', 'mean'),
              pct_cursa_2025=('cursa_2025', 'mean'))
         .round(2).sort_values('nota_promedio'))

**Lectura:**
- La etiqueta es **coherente con la conducta**: los ELIMINADOS tienen nota promedio ~3,3 y más de la mitad de sus evaluaciones con nota mínima, contra ~16% en los VIGENTES. SUSPENDIDO queda en un punto intermedio.
- La opción A (ausencia en rendimiento) **no sirve como etiqueta por sí sola**: cerca del 45% de los VIGENTES tampoco aparece en 2025-P1. El extracto mezcla regímenes y cortes, así que "no tener notas" no equivale a desertar.

### 3.4.3 Validación con contactabilidad

In [ ]:
log = pd.read_pickle('data/interim/contactabilidad_log_limpio.pkl')
SENALES = ['TX_NO_CONTESTA', 'TX_DESISTE', 'TX_NO_CONTINUA', 'TX_NO_MATRICULADO',
           'TX_FINANCIERO', 'TX_SIN_ENTREGAS', 'TX_INASISTENCIA', 'TX_CONTINUA']
contacto_alumno = log.groupby('RUT_ALUMNO').agg(n_contactos=('FECHA_COMUNICO', 'size'),
                                                **{s: (s, 'max') for s in SENALES})
val_cont = estado_alumno.join(contacto_alumno, how='inner')
print('Encuestados con al menos un contacto:', len(val_cont), 'de', len(estado_alumno))
print('Contactados SIN encuesta (sin etiqueta disponible):', len(set(contacto_alumno.index) - set(estado_alumno.index)))

# % de estudiantes que tuvo la señal alguna vez, por estado
val_cont.groupby('estado')[['n_contactos'] + SENALES].mean().round(2).T

**Lectura:**
- Las señales explícitas acompañan a la etiqueta: `TX_DESISTE` aparece en el 18% de los ELIMINADOS y el 43% de los SUSPENDIDOS, contra el 6% de los VIGENTES. Lo mismo pasa con `TX_NO_CONTINUA` y `TX_NO_MATRICULADO`.
- Esas señales son **contemporáneas o posteriores** a la deserción. Usarlas como predictores sin corte temporal produce **fuga de información (leakage)**.
- `TX_NO_CONTESTA` casi no discrimina en la forma "alguna vez" (64% contra 59%). Su valor predictivo debería estar en la **frecuencia y la tendencia reciente**, no en si ocurrió alguna vez.
- 3.588 estudiantes contactados no aparecen en ninguna encuesta, así que **no tienen etiqueta**. La población etiquetada se limita a quienes respondieron la encuesta.

### 3.4.4 Definición propuesta de la variable objetivo

| Estado académico | `DESERCION` (principal) | `DESERCION_ESTRICTA` (sensibilidad) |
|---|---|---|
| ELIMINADO | 1 | 1 |
| SUSPENDIDO | 1 | excluido |
| VIGENTE | 0 | 0 |
| TITULADO / EGRESADO | 0 | 0 |
| SIN DATO | excluido | excluido |

- **Principal:** desertor = salió del sistema, ya sea de forma definitiva (ELIMINADO) o temporal (SUSPENDIDO). Para alertas tempranas conviene tratarlos igual, porque requieren la misma acción de retención.
- **Estricta:** solo ELIMINADO contra el resto. Sirve para comprobar que las conclusiones no dependen de cómo se clasifica a los suspendidos.
- **`meses_exposicion`:** meses entre la respuesta de la encuesta y la fecha de corte. Como el estado no tiene fecha, hay que controlar esta variable (o estratificar por cohorte) para no confundir "más tiempo" con "más riesgo".

> **SUPUESTO:** `FECHA_CORTE = 2026-08-31` (fin del log de contactabilidad). Hay que reemplazarla por la fecha real del extracto de `Estado académico`.

In [ ]:
FECHA_CORTE = pd.Timestamp('2026-08-31')   # SUPUESTO: confirmar con Registro Académico

MAPA_PRINCIPAL = {'ELIMINADO': 1, 'SUSPENDIDO': 1, 'VIGENTE': 0, 'TITULADO': 0, 'EGRESADO': 0}
MAPA_ESTRICTO = {'ELIMINADO': 1, 'VIGENTE': 0, 'TITULADO': 0, 'EGRESADO': 0}

base = enc.sort_values('cohorte').drop_duplicates('token', keep='first').copy()
base['DESERCION'] = base['estado'].map(MAPA_PRINCIPAL)              # SIN DATO -> NaN
base['DESERCION_ESTRICTA'] = base['estado'].map(MAPA_ESTRICTO)      # SUSPENDIDO y SIN DATO -> NaN
base['meses_exposicion'] = ((FECHA_CORTE - base['fecha_encuesta']).dt.days / 30.44).round(1)
base['tiene_rendimiento'] = base['token'].isin(rend_alumno.index)
base['tiene_contacto'] = base['token'].isin(contacto_alumno.index)

resumen = base.groupby('cohorte').agg(
    estudiantes=('token', 'size'),
    con_etiqueta=('DESERCION', 'count'),
    tasa_desercion=('DESERCION', 'mean'),
    tasa_desercion_estricta=('DESERCION_ESTRICTA', 'mean'),
    meses_exposicion_mediana=('meses_exposicion', 'median'),
    pct_con_rendimiento=('tiene_rendimiento', 'mean'),
    pct_con_contacto=('tiene_contacto', 'mean'),
)
total = pd.DataFrame({'estudiantes': [len(base)], 'con_etiqueta': [base['DESERCION'].count()],
                      'tasa_desercion': [base['DESERCION'].mean()],
                      'tasa_desercion_estricta': [base['DESERCION_ESTRICTA'].mean()],
                      'meses_exposicion_mediana': [base['meses_exposicion'].median()],
                      'pct_con_rendimiento': [base['tiene_rendimiento'].mean()],
                      'pct_con_contacto': [base['tiene_contacto'].mean()]}, index=['Total'])
pd.concat([resumen, total]).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3))
x = resumen.index.astype(str)
ax.bar(x, resumen['tasa_desercion'] * 100, color='#B8433E', width=0.55)
for xi, (t, n, m) in enumerate(zip(resumen['tasa_desercion'], resumen['con_etiqueta'], resumen['meses_exposicion_mediana'])):
    ax.text(xi, t * 100 + 1.2, f'{t*100:.1f}%', ha='center', fontsize=10, fontweight='bold', color='#333333')
    ax.text(xi, 2, f'n = {n}\n~{m:.0f} meses', ha='center', fontsize=8, color='white')
ax.set_ylabel('% deserción (eliminado + suspendido)')
ax.set_ylim(0, max(resumen['tasa_desercion']) * 100 + 10)
ax.set_title('Tasa de deserción por cohorte (variable DESERCION)', loc='left', fontsize=11)
ax.yaxis.grid(True, color='#EEEEEE'); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

## 3.5 Tabla base con la variable objetivo
Una fila por estudiante encuestado, con la etiqueta, el tiempo de exposición y la disponibilidad de datos en las otras fuentes. Es la tabla a la que se unirán los predictores en la fase 4.

In [ ]:
# Tabla base con la variable objetivo (una fila por estudiante). No se versiona: *.pkl está en .gitignore
base_objetivo = base[['token', 'cohorte', 'fecha_encuesta', 'carrera', 'jornada', 'estado',
                      'DESERCION', 'DESERCION_ESTRICTA', 'meses_exposicion',
                      'tiene_rendimiento', 'tiene_contacto']]
base_objetivo.to_pickle('data/processed/base_objetivo.pkl')
print(base_objetivo.shape)
print(base_objetivo['DESERCION'].value_counts(dropna=False))

# 4. Modelado

**Estado:** planificado. Se puede construir un modelo de referencia con los datos actuales, pero la versión definitiva depende de las solicitudes R1 y R2 (fechas).

## 4.1 Unidad de análisis y momento de predicción
- **Unidad:** estudiante (una fila por estudiante, tabla `base_objetivo`).
- **Modelo A, al ingreso:** solo encuesta de caracterización. Aplicable a todas las cohortes.
- **Modelo B, al cierre del primer periodo:** encuesta, notas del primer periodo y contactos del primer periodo. Hoy solo es posible para la cohorte 2024.

## 4.2 Predictores candidatos y regla temporal
| Fuente | Predictores candidatos | Regla |
|---|---|---|
| Encuesta | Antecedentes personales, familiares y socioeconómicos, trayectoria formativa y laboral, motivación, bienestar, disposición hacia la educación superior | Disponibles al ingreso; sin restricción temporal |
| Rendimiento | Nota promedio, % reprobadas, % nota mínima, n° de evaluaciones del primer periodo | Solo el primer periodo |
| Contactabilidad | N° de contactos, % no contesta, inicia el estudiante, menciones financieras, inasistencia, sin entregas | Solo contactos anteriores al cierre del primer periodo |
| **Excluidos** | `TX_DESISTE`, `TX_NO_CONTINUA`, `TX_NO_MATRICULADO`, `CAT_ASUNTO = RETENCION` | Contemporáneos a la deserción (leakage) |

## 4.3 Técnicas
1. **Línea base:** regresión logística con pesos de clase (las clases están desbalanceadas, ≈ 30% / 70%). Es interpretable y sirve de referencia.
2. **Modelos de comparación:** árboles de decisión y gradient boosting.
3. **Variables sensibles:** se evaluará su efecto y su sesgo antes de decidir si entran al modelo (fase 5).

## 4.4 Diseño de validación
- **Por cohorte:** entrenar con 2024–2025 y probar con 2026, para simular el uso real con una cohorte nueva.
- **Validación cruzada estratificada** dentro del conjunto de entrenamiento, para ajustar hiperparámetros.
- **Exposición:** controlar `meses_exposicion` o estratificar por cohorte mientras no exista una etiqueta con horizonte fijo.
- **Sensibilidad:** repetir con `DESERCION_ESTRICTA`.

## 4.5 Dependencias
| Solicitud | Qué cambia en el modelado |
|---|---|
| R1 · Historial de estados con fecha | Etiqueta a horizonte fijo (p. ej., deserción a 12 meses) y corte temporal exacto de los predictores |
| R2 · Fecha de corte del estado | Cálculo correcto de `meses_exposicion` |
| R4 · Estado de toda la matrícula | Más población etiquetada y control del sesgo de quienes responden la encuesta |
| R5 · Notas desde 2025-P2 | Modelo B para las cohortes 2025 y 2026 |

# 5. Evaluación

**Estado:** preliminar (21-11-2026). Criterios definidos de antemano para no ajustarlos a los resultados; resultados en la sección 5.1.

| Dimensión | Métrica o revisión | Por qué |
|---|---|---|
| Discriminación | PR-AUC (y ROC-AUC como referencia) | Con ≈ 30% de positivos, PR-AUC refleja mejor la utilidad |
| Priorización | Recall y precisión en el 10% y el 20% de mayor riesgo | Es lo que el equipo de seguimiento puede atender |
| Calibración | Curva de calibración, Brier score | Las probabilidades deben poder leerse como riesgo |
| Anticipación | Meses entre la alerta y la deserción | Objetivo del negocio (requiere R1) |
| Estabilidad | Desempeño por cohorte, carrera y jornada | Evitar un modelo que solo funcione en una cohorte |
| Equidad | Desempeño y tasa de alerta por variables sensibles | Criterio ético (fase 1.4) |
| Negocio | Revisión de casos y de factores de riesgo con InnovaLab | Validar que tenga sentido para quien lo usa |

Si el modelo no cumple los criterios de la fase 1, se vuelve a la fase 3 (nuevos predictores) o a la fase 1 (redefinir el objetivo).

## 5.1 Resultados preliminares del modelo de referencia

El entrenamiento y la evaluación están en `src/fuga_ipss/modelo.py` (`python -m fuga_ipss.modelo`). Los resultados agregados quedan en `reports/resultados_modelo.json` y los gráficos en `reports/figures/`. El análisis completo está en `docs/02_informe_proyecto/Informe_Proyecto_FUGA_IPSS.docx`.

| | Modelo A · al ingreso | Modelo B · cierre del 1er periodo |
|---|---|---|
| Validación | Temporal: 2024–2025 → 2026 | CV 5 particiones, cohorte 2024 |
| Predictores | 178 de la encuesta (reglas sobre el diccionario, sin atributos protegidos) | A + notas y contactos del primer periodo (sin leakage) |
| AUC | 0,57 (IC 95% 0,54–0,60) | 0,75 (A en la misma cohorte: 0,58) |
| Captura en el 20% de mayor riesgo | 25% (al azar: 20%) | 41% (A: 24%) |
| % de alertados que desertó | 39% (base 31%) | 85% (base 42%) |

**Lectura:**
- La encuesta de ingreso tiene una señal real pero débil, y es estable entre cohortes (captura del 24% al 27% en el 20% superior en la CV dentro de cada cohorte). El gradient boosting no mejora a la regresión logística: el límite está en la información, no en la técnica.
- La conducta del primer periodo casi duplica la captura. Esta cifra es optimista mientras no exista la fecha de deserción (R1).
- **Equidad:** con el corte del 20%, el Modelo A alerta al 37% de la jornada vespertina y al 13% de la online, con tasas de deserción parecidas, y alerta más a mujeres y extranjeros. Hay que corregirlo antes de cualquier uso individual.


# 6. Despliegue

**Estado:** pendiente. Propuesta inicial, a validar con InnovaLab.

- **Producto:** al cierre de cada periodo, una lista priorizada de estudiantes en riesgo para el equipo de seguimiento, con los principales factores de riesgo de cada uno y sin exponer datos sensibles.
- **Integración:** a definir con el área de TI (exportación periódica o integración con el SGE).
- **Monitoreo:** comparar las alertas con las deserciones reales cada periodo y vigilar cambios en los datos (por ejemplo, cambios en la encuesta).
- **Reentrenamiento:** al menos una vez por año, con la cohorte nueva.
- **Gobernanza:** acceso restringido a la lista, registro de uso y revisión periódica de sesgos.

# Anexo. Registro de decisiones, problemas abiertos y solicitudes

## Decisiones
| Fecha | Decisión | Fase |
|---|---|---|
| 10-11-2026 | Variable objetivo `DESERCION` = ELIMINADO o SUSPENDIDO (1) frente a VIGENTE, TITULADO o EGRESADO (0); SIN DATO excluido | 3 |
| 10-11-2026 | Versión estricta `DESERCION_ESTRICTA` (solo ELIMINADO) para análisis de sensibilidad | 3 |
| 10-11-2026 | Estudiantes repetidos entre cohortes: se conserva la primera cohorte | 3 |
| 10-11-2026 | Señales TX_DESISTE, TX_NO_CONTINUA y TX_NO_MATRICULADO excluidas como predictores | 4 |
| 17-11-2026 | Notebook reorganizado según CRISP-DM | — |
| 21-11-2026 | Modelo de referencia y evaluación preliminar (ver `docs/registro_decisiones.md`) | 4–5 |

## Problemas abiertos
1. El estado académico no trae **fecha de corte** ni **fecha del evento**, así que no se puede fijar un horizonte de predicción ni cortar los predictores en el tiempo.
2. **Exposición desigual** entre cohortes (≈ 31, 18 y 7,5 meses): hay que controlar `meses_exposicion` o modelar por cohorte.
3. Falta el **significado institucional** de ELIMINADO y SUSPENDIDO, y la explicación de por qué la cohorte 2026 tiene más eliminados que la 2025.
4. **Cobertura:** solo los encuestados tienen etiqueta; 3.588 estudiantes contactados quedan fuera.
5. **Leakage:** las señales contemporáneas a la deserción no pueden ser predictores sin una fecha de corte.

## Solicitudes a Registro Académico
| N° | Solicitud | Prioridad |
|---|---|---|
| R1 | Historial de cambios de estado académico con fecha (matrícula 2024–2026) | Crítica |
| R2 | Fecha de extracción del Estado académico en cada archivo de encuesta | Crítica |
| R3 | Definición oficial de los estados y explicación de la tasa de la cohorte 2026 | Alta |
| R4 | Estado académico de toda la matrícula | Alta |
| R5 | Notas desde 2025-P2 (incluida la cohorte 2026) | Alta |
| R6 | Tabla de equivalencias entre códigos y nombres de carrera | Media |

El detalle está en `docs/04_documentos_tecnicos/`.